### N-gram language models or how to write scientific papers (4 pts)

We shall train our language model on a corpora of [ArXiv](http://arxiv.org/) articles and see if we can generate a new one!

![img](https://media.npr.org/assets/img/2013/12/10/istock-18586699-monkey-computer_brick-16e5064d3378a14e0e4c2da08857efe03c04695e-s800-c85.jpg)

_data by neelshah18 from [here](https://www.kaggle.com/neelshah18/arxivdataset/)_

_Disclaimer: this has nothing to do with actual science. But it's fun, so who cares?!_

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# Alternative manual download link: https://yadi.sk/d/_nGyU2IajjR9-w
!wget "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" -O arxivData.json.tar.gz
!tar -xvzf arxivData.json.tar.gz
data = pd.read_json("./arxivData.json")
data.sample(n=5)

--2026-01-13 20:34:45--  https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.65.18, 2620:100:6021:18::a27d:4112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.65.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/0mulrothty5o8i8ud9gz2/arxivData.json.tar.gz?rlkey=n759u5qx2xpxxglmrl390vwvk&dl=1 [following]
--2026-01-13 20:34:46--  https://www.dropbox.com/scl/fi/0mulrothty5o8i8ud9gz2/arxivData.json.tar.gz?rlkey=n759u5qx2xpxxglmrl390vwvk&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc755d746b80dab497b8349603e2.dl.dropboxusercontent.com/cd/0/inline/C46bFqpzzMZRqWMf0n9TUGbu7dp0L5EyqCMCLU8aR_obzLMiGUG5XUQ2Z9I2OPsYnqUbis5s_8VnEEBQesxtDcmD0mPlqg4zRGWktRr1nG2Yp8SrNd77DTtObE5bxAN8Ti8/file?dl=1# [following]
--2026-01-13 20:34:46--  https://uc755d746b80dab497b8349603e2.dl.dropbox

,author,day,id,link,month,summary,tag,title,year
19313,"[{'name': 'Masaki Togai'}, {'name': 'Hiroyuki ...",27,1304.3112v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",3,The role of inferencing with uncertainty is be...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",A VLSI Design and Implementation for a Real-Ti...,2013
34695,[{'name': 'Paul Christian Sommerhoff'}],26,1709.09119v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",9,If someone is looking for a certain publicatio...,"[{'term': 'cs.CL', 'scheme': 'http://arxiv.org...",Integration of Japanese Papers Into the DBLP D...,2017
13778,"[{'name': ""Vladimir G. Red'ko""}]",18,1411.5053v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",11,The model of interaction between learning and ...,"[{'term': 'cs.NE', 'scheme': 'http://arxiv.org...",Model of Interaction between Learning and Evol...,2014
8013,"[{'name': 'Jeff Zhang'}, {'name': 'Kartheek Ra...",11,1802.03806v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",2,Hardware accelerators are being increasingly d...,"[{'term': 'cs.NE', 'scheme': 'http://arxiv.org...",ThUnderVolt: Enabling Aggressive Voltage Under...,2018
16796,"[{'name': 'Lawrence Phillips'}, {'name': 'Nath...",6,1706.01839v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",6,"Increasingly, cognitive scientists have demons...","[{'term': 'cs.CL', 'scheme': 'http://arxiv.org...",Assessing the Linguistic Productivity of Unsup...,2017


In [3]:
# assemble lines: concatenate title and description
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

### Tokenization

You know the dril. The data is messy. Go clean the data. Use WordPunctTokenizer or something.


In [4]:
# Task: convert lines (in-place) into strings of space-separated tokens. Import & use WordPunctTokenizer

from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer()
for index in range(len(lines)):
    lines[index] = ' '.join(tokenizer.tokenize(lines[index])).lower()

lines[:3]

['dual recurrent attention units for visual question answering ; we propose an architecture for vqa which utilizes recurrent layers to generate visual and textual attention . the memory characteristic of the proposed recurrent attention units offers a rich joint embedding of visual and textual features and enables the model to reason relations between several parts of the image and question . our single model outperforms the first place winner on the vqa 1 . 0 dataset , performs within margin to the current state - of - the - art ensemble model . we also experiment with replacing attention mechanisms in other state - of - the - art models with our implementation and show increased accuracy . in both cases , our recurrent attention mechanism improves performance in tasks requiring sequential or relational reasoning on the vqa dataset .',
 'sequential short - text classification with recurrent and convolutional neural networks ; recent approaches based on artificial neural networks ( ann

In [5]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

### N-Gram Language Model (1point)

A language model is a probabilistic model that estimates text probability: the joint probability of all tokens $w_t$ in text $X$: $P(X) = P(w_1, \dots, w_T)$.

It can do so by following the chain rule:
$$P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1}).$$

The problem with such approach is that the final term $P(w_T \mid w_1, \dots, w_{T-1})$ depends on $n-1$ previous words. This probability is impractical to estimate for long texts, e.g. $T = 1000$.

One popular approximation is to assume that next word only depends on a finite amount of previous words:

$$P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})$$

Such model is called __n-gram language model__ where n is a parameter. For example, in 3-gram language model, each word only depends on 2 previous words.

$$
    P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

You can also sometimes see such approximation under the name of _n-th order markov assumption_.

The first stage to building such a model is counting all word occurences given N-1 previous words

In [ ]:
from tqdm import tqdm
from collections import defaultdict, Counter

# special tokens:
# - `UNK` represents absent tokens,
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - if prefix is shorter than (n - 1) tokens, it should be padded with UNK. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.
    """
    counts = defaultdict(Counter)
    # counts[(word1, word2)][word3] = how many times word3 occured after (word1, word2)

    for line in lines:
        line_tokens = line.split()
        processed_line = [UNK] * (n - 1) + line_tokens + [EOS]

        for i in range(n - 1, len(processed_line)):
            # The prefix is the (n-1) tokens immediately preceding the current token
            prefix = tuple(processed_line[i - (n - 1) : i])
            current_token = processed_line[i]
            counts[prefix][current_token] += 1

    return counts

In [ ]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=2)
# assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
# assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
# assert dummy_counts['_UNK_', 'a']['note'] == 3
# assert dummy_counts['p', '=']['np'] == 2
# assert dummy_counts['author', '.']['_EOS_'] == 1

In [ ]:
dummy_counts

defaultdict(collections.Counter,
            {('_UNK_',): Counter({'differential': 1,
                      'what': 1,
                      'p': 1,
                      'computational': 1,
                      'weak': 1,
                      'creating': 1,
                      'defeasible': 1,
                      'essence': 1,
                      'deep': 1,
                      'statistical': 1,
                      'complex': 1,
                      'serious': 1,
                      'preprocessing': 1,
                      'liquid': 1,
                      'mining': 1,
                      'towards': 1,
                      'a': 13,
                      'icon': 1,
                      'recognition': 1,
                      'glottochronologic': 1,
                      'the': 3,
                      'utility': 1,
                      'temporized': 1,
                      'backpropagation': 1,
                      'random': 1,
                      'network': 1,

Once we can count N-grams, we can build a probabilistic language model.
The simplest way to compute probabilities is in proporiton to counts:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [ ]:
lines[0]

'dual recurrent attention units for visual question answering ; we propose an architecture for vqa which utilizes recurrent layers to generate visual and textual attention . the memory characteristic of the proposed recurrent attention units offers a rich joint embedding of visual and textual features and enables the model to reason relations between several parts of the image and question . our single model outperforms the first place winner on the vqa 1 . 0 dataset , performs within margin to the current state - of - the - art ensemble model . we also experiment with replacing attention mechanisms in other state - of - the - art models with our implementation and show increased accuracy . in both cases , our recurrent attention mechanism improves performance in tasks requiring sequential or relational reasoning on the vqa dataset .'

In [ ]:
class NGramLanguageModel:
    def __init__(self, lines, n):
        """
        Train a simple count-based language model:
        compute probabilities P(w_t | prefix) given ngram counts

        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n

        counts = count_ngrams(lines, self.n)

        # compute token proabilities given counts
        self.probs = defaultdict(Counter)
        # probs[(word1, word2)][word3] = P(word3 | word1, word2)

        # populate self.probs with actual probabilities
        for prefix, token_counts in counts.items():
            total_count = sum(token_counts.values())
            for token, count in token_counts.items():
                self.probs[prefix][token] = count / total_count


    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        prefix_tokens = prefix.split()
        # Only take the last (n-1) tokens for the prefix
        prefix_tokens = prefix_tokens[max(0, len(prefix_tokens) - (self.n - 1)):]
        # Pad with UNK if the prefix is shorter than (n-1)
        prefix_tokens = [UNK] * (self.n - 1 - len(prefix_tokens)) + prefix_tokens
        return self.probs[tuple(prefix_tokens)]

    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

In [ ]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

In [ ]:
dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']

Counter({'differential': 0.01,
         'what': 0.01,
         'p': 0.01,
         'computational': 0.01,
         'weak': 0.01,
         'creating': 0.01,
         'defeasible': 0.01,
         'essence': 0.01,
         'deep': 0.01,
         'statistical': 0.01,
         'complex': 0.01,
         'serious': 0.01,
         'preprocessing': 0.01,
         'liquid': 0.01,
         'mining': 0.01,
         'towards': 0.01,
         'a': 0.13,
         'icon': 0.01,
         'recognition': 0.01,
         'glottochronologic': 0.01,
         'the': 0.03,
         'utility': 0.01,
         'temporized': 0.01,
         'backpropagation': 0.01,
         'random': 0.01,
         'network': 0.01,
         'glottochronology': 0.01,
         'using': 0.02,
         'time': 0.01,
         'convolutional': 0.01,
         'fitness': 0.01,
         'flip': 0.01,
         'autonomous': 0.01,
         'activitynet': 0.01,
         'decision': 0.01,
         'text': 0.01,
         'discrimination': 0.01,


Let's test it!

In [ ]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

Now that you've got a working n-gram language model, let's see what sequences it can generate. But first, let's train it on the whole dataset.

In [ ]:
lm = NGramLanguageModel(lines, n=3)

In [ ]:
prefix = "deep variational"
while prefix[-5:] != EOS:
    try:
        tokens, probs = zip(*lm.get_possible_next_tokens(prefix).items())
    except ValueError:
        prefix += EOS
        continue
    prefix += (' ' + tokens[np.argmax(probs)])
    if len(prefix) > 100:
        prefix += EOS
        break
print(prefix[:-5])

deep variational auto - encoder ( vae ) with a single image super - resolution ( sr ) is a challenging


The process of generating sequences is... well, it's sequential. You maintain a list of tokens and iteratively add next token by sampling with probabilities.

$ X = [] $

__forever:__
* $w_{next} \sim P(w_{next} | X)$
* $X = concat(X, w_{next})$


Instead of sampling with probabilities, one can also try always taking most likely token, sampling among top-K most likely tokens or sampling with temperature. In the latter case (temperature), one samples from

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Where $\tau > 0$ is model temperature. If $\tau << 1$, more likely tokens will be sampled with even higher probability while less likely tokens will vanish.

In [ ]:
def get_next_token(lm, prefix, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """
    probs = np.array(list(
        lm.get_possible_next_tokens(prefix).values()
        )
    )
    tokens = list(
        lm.get_possible_next_tokens(prefix).keys()
    )
    if temperature == 0:
        return tokens[np.argmax(probs)]

    return tokens[
        np.random.choice(
            len(probs),
            p=probs ** (1 / temperature) / np.sum(probs ** (1 / temperature))
        )
    ]


In [ ]:
get_next_token(lm, "there have", 0.4)

'been'

In [ ]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000

print("Looks nice!")

Looks nice!


Let's have fun with this model

In [ ]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

artificial intelligence . instead , we show that information loss in the context of osns . _EOS_


In [ ]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

bridging the gap between the source of information , i . e ., the number of samples . _EOS_


__More in the homework:__ nucleus sampling, top-k sampling, beam search(not for the faint of heart).

### Evaluating language models: perplexity (1point)

Perplexity is a measure of how well your model approximates the true probability distribution behind the data. __Smaller perplexity = better model__.

To compute perplexity on one sentence, use:
$$
    {\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$


On the corpora level, perplexity is a product of probabilities of all tokens in all sentences to the power of $1/N$, where $N$ is __total length (in tokens) of all sentences__ in corpora.

This number can quickly get too small for float32/float64 precision, so we recommend you to first compute log-perplexity (from log-probabilities) and then take the exponent.

In [ ]:
np.log(23)

np.float64(3.1354942159291497)

In [ ]:
import numpy as np

def perplexity(lm, lines, min_logprob=np.log(10 ** -50.)):
    """
    :param lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above

    Note: do not forget to compute P(w_first | empty) and P(eos | full_sequence)

    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    log_probs_sum = 0.0
    total_actual_tokens = 0

    # Iterate through each line to calculate its contribution to perplexity independently
    for line in lines:
        if not line.strip():  # Skip empty lines to avoid errors
            continue

        # Prepare the sequence for the current line:
        # 1. Prepend (n-1) UNK tokens for proper n-gram prefixes at the beginning of the sentence.
        # 2. Split the line into its individual tokens.
        # 3. Append the special EOS (End-Of-Sentence) token.
        current_line_tokens = line.split()
        effective_sequence = [UNK] * (lm.n - 1) + current_line_tokens + [EOS]

        # N is the total length in tokens of all sentences in the corpora.
        # For each sentence, N includes its actual tokens and the EOS token, but not the UNK padding.
        total_actual_tokens += (len(current_line_tokens) + 1)

        # Iterate through the effective_sequence to predict each token and accumulate its log probability.
        # The loop starts from index (lm.n - 1) because the first (lm.n - 1) tokens are UNK padding,
        # and the first prediction is for the token at index (lm.n - 1) given its preceding (lm.n - 1) tokens.
        for i in range(lm.n - 1, len(effective_sequence)):
            current_token = effective_sequence[i]

            # Construct the prefix for the current token:
            # It consists of the (lm.n - 1) tokens immediately preceding the current_token.
            prefix_tokens = effective_sequence[i - (lm.n - 1) : i]
            prefix_string = ' '.join(prefix_tokens)

            # Get the probability of predicting the current_token given its prefix.
            prob = lm.get_next_token_prob(prefix_string, current_token)

            # Calculate the log probability.
            # If the probability is zero, it indicates an unseen n-gram, which would result in log(0) = -infinity.
            # To handle this, we use min_logprob as a floor value.
            log_prob = np.log(prob) if prob > 0 else min_logprob

            # Accumulate the log probabilities, ensuring no log_prob falls below min_logprob
            # (even for very small non-zero probabilities, to prevent numerical underflow if np.log(prob) < min_logprob).
            log_probs_sum += max(log_prob, min_logprob)

    # Handle the case where no tokens were processed (e.g., empty 'lines' list).
    if total_actual_tokens == 0:
        return float('inf')

    # Compute the perplexity using the formula: exp(-1/N * sum(log P)).
    perplexity_value = np.exp(-log_probs_sum / total_actual_tokens)
    return perplexity_value

In [ ]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)

# Create a sentence with words likely not in dummy_lines to test min_logprob effect
ppx_missing = perplexity(lm3, ["this is a very very unique sentence with unseen words"])

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f ppx_missing=%.3f" % (ppx1, ppx3, ppx10, ppx_missing))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249), atol=1e-5), "Perplexities do not match expected values. Check calculations or min_logprob."

Perplexities: ppx1=318.213 ppx3=1.520 ppx10=1.184 ppx_missing=2983413995330796354525038050474649322520576000.000


Now let's measure the actual perplexity: we'll split the data into train and test and score model on test data only.

In [ ]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))


N = 1, Perplexity = 1832.23136
N = 2, Perplexity = 85653987.28774
N = 3, Perplexity = 61999196259043346743296.00000


In [ ]:
# whoops, it just blew up :)

### LM Smoothing

The problem with our simple language model is that whenever it encounters an n-gram it has never seen before, it assigns it with the probabilitiy of 0. Every time this happens, perplexity explodes.

To battle this issue, there's a technique called __smoothing__. The core idea is to modify counts in a way that prevents probabilities from getting too low. The simplest algorithm here is Additive smoothing (aka [Lapace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

If counts for a given prefix are low, additive smoothing will adjust probabilities to a more uniform distribution. Not that the summation in the denominator goes over _all words in the vocabulary_.

Here's an example code we've implemented for you:

In [ ]:
class LaplaceLanguageModel(NGramLanguageModel):
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}

    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total) # prevent rounding errors
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))


**Disclaimer**: the implementation above assumes all words unknown within a given context to be equally likely, *as well as the words outside of vocabulary*. Therefore, its' perplexity will be lower than it should when encountering such words. Therefore, comparing it with a model with fewer unknown words will not be fair. When implementing your own smoothing, you may handle this by adding a virtual `UNK` token of non-zero probability. Technically, this will result in a model where probabilities do not add up to $1$, but it is close enough for a practice excercise.

In [ ]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [ ]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

N = 1, Perplexity = 1832.66878
N = 2, Perplexity = 470.48021
N = 3, Perplexity = 3679.44765


In [ ]:
# optional: try to sample tokens from such a model

### Kneser-Ney smoothing (2 points)

Additive smoothing is simple, reasonably good but definitely not a State of The Art algorithm.


Your final task in this notebook is to implement [Kneser-Ney](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing) smoothing.

It can be computed recurrently, for n>1:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

where
- $prefix_{n-1}$ is a tuple of {n-1} previous tokens
- $lambda_{prefix_{n-1}}$ is a normalization constant chosen so that probabilities add up to 1
- Unigram $P_{kn}(w_t | prefix_{n-2})$ corresponds to Kneser Ney smoothing for {N-1}-gram language model.
- Unigram $P_{kn}(w_t)$ is a special case: how likely it is to see x_t in an unfamiliar context

See lecture slides or wiki for more detailed formulae.

__Your task__ is to
- implement `KneserNeyLanguageModel` class,
- test it on 1-3 gram language models
- find optimal (within reason) smoothing delta for 3-gram language model with Kneser-Ney smoothing

In [ ]:
from IPython.lib.security import passwd
class KneserNeyLanguageModel(NGramLanguageModel):
    """ A template for Kneser-Ney language model. Default delta may be suboptimal. """
    def __init__(self, lines, n, delta=1.0):
        self.n = n

    def get_possible_next_tokens(self, prefix):
        passwd

    def get_next_token_prob(self, prefix, next_token):
        pass

In [ ]:
import numpy as np
from collections import defaultdict, Counter

# UNK and EOS are defined globally in the notebook context
# UNK, EOS = "_UNK_", "_EOS_"

class KneserNeyLanguageModel(NGramLanguageModel):
    """
    Kneser-Ney language model implementation.
    Calculates probabilities using a recursive formula with discounting.
    """
    def __init__(self, lines, n, delta=0.75):
        self.n = n
        self.delta = delta

        # self.counts[k] stores k-gram counts: {prefix_k-1: {token: count}}
        self.counts = {}
        # self.N_1_plus_prefix[k][prefix_k-1] stores N_1+(prefix_k-1),
        # which is the number of unique words that follow prefix_k-1 to form a k-gram.
        # This is used in the lambda calculation.
        self.N_1_plus_prefix = {}
        # self.unigram_continuation_counts[word] stores C_cont(word),
        # the number of unique words that precede 'word' in a bigram.
        # This is used for the Kneser-Ney unigram base probability (P_kn(w)).
        self.unigram_continuation_counts = Counter()

        # Stores all observed tokens (excluding UNK, including EOS).
        self.vocab = set()

        # 1. Compute all k-gram counts for k from 1 to n
        for k in range(1, n + 1 + int(n == 1)):
            current_k_counts = count_ngrams(lines, k)
            self.counts[k] = current_k_counts
            if k == 1:
                # Add all tokens to vocab from 1-gram counts (excluding UNK padding token itself)
                self.vocab.update(token for token in current_k_counts[tuple()] if token != UNK)
            elif k > 1:
                # Compute N_1+(prefix_{k-1}) for current k-grams
                # This is the number of unique types that follow prefix_{k-1}.
                self.N_1_plus_prefix[k] = {
                    prefix_k_1: len(next_token_counts)
                    for prefix_k_1, next_token_counts in current_k_counts.items()
                }

        # 2. Compute C_cont(word) for Kneser-Ney unigram probability (P_kn(w))
        # This is based on bigram counts (self.counts[2])
        # C_cont(w) = count of unique predecessors of w in a bigram.
        distinct_predecessors_for_word = defaultdict(set)
        if 2 in self.counts:  # Ensure bigrams exist to compute continuation counts for unigrams
            for (w_prev,), next_word_counts in self.counts[2].items():
                for w_curr in next_word_counts:
                    distinct_predecessors_for_word[w_curr].add(w_prev)

        for word, predecessors_set in distinct_predecessors_for_word.items():
            # Кол-во уникальных подходящих биграм
            self.unigram_continuation_counts[word] = len(predecessors_set)

        # Кол-во уникальных биграм
        self.total_unigram_continuation_sum = sum(self.unigram_continuation_counts.values())

        # 3. Precompute total counts for each prefix (denominator for the first term and lambda)
        # self.prefix_total_counts[k][prefix_k-1] = Sum_w Count(prefix_k-1, w)
        self.prefix_total_counts = defaultdict(lambda: defaultdict(int))
        for k in range(1, n + 1):
            for prefix_k_1, next_tokens_counts in self.counts[k].items():
                self.prefix_total_counts[k][prefix_k_1] = sum(next_tokens_counts.values())

        # Add EOS and UNK to vocab if they aren't already, for consistency in get_possible_next_tokens
        self.vocab.add(EOS)
        self.vocab.add(UNK)

    def _get_unigram_kn_prob(self, next_token):
        """
        Для n == 1
        Base case for Kneser-Ney recursion: P_kn(w_t).
        P_kn(w_t) = C_cont(w_t) / Sum_{w'} C_cont(w')
        """
        if self.total_unigram_continuation_sum == 0:
            raise ValueError("division by zero for empty corpus or no bigrams")
            return 0.0  # Avoid division by zero for empty corpus or no bigrams
        return self.unigram_continuation_counts.get(next_token, 0) / self.total_unigram_continuation_sum

    def _get_next_token_prob_recursive(self, prefix_str, next_token, current_n_order):
        """
        Helper function to calculate Kneser-Ney probability for a specific order (current_n_order).
        This function is called recursively.
        """
        # Base case for recursion: unigram Kneser-Ney (P_kn(w_t))
        if current_n_order == 1:
            return self._get_unigram_kn_prob(next_token)

        # Prepare prefix for this specific order
        # The prefix for a `current_n_order` model is `current_n_order - 1` tokens long.
        prefix_tokens_from_str = prefix_str.split()
        effective_prefix_length = current_n_order - 1

        # Extract the relevant part of the prefix string for this order.
        # This handles cases where the provided `prefix_str` is shorter than the full n-1 context
        # by taking all available tokens, and padding will add UNKs as needed.
        current_prefix_tuple_unpadded = tuple(prefix_tokens_from_str[max(0, len(prefix_tokens_from_str) - effective_prefix_length):])

        # Pad with UNK if the prefix is shorter than what's needed for this order.
        # This handles initial sentence prefixes like "" or "word".
        if len(current_prefix_tuple_unpadded) < effective_prefix_length:
            padded_prefix_list = [UNK] * (effective_prefix_length - len(current_prefix_tuple_unpadded)) + list(current_prefix_tuple_unpadded)
            current_prefix_tuple = tuple(padded_prefix_list)
        else:
            current_prefix_tuple = current_prefix_tuple_unpadded

        # Get raw counts for the current order's n-gram
        ngram_counts_for_prefix = self.counts[current_n_order].get(current_prefix_tuple, Counter())
        count_prefix_next_token = ngram_counts_for_prefix.get(next_token, 0)

        # Denominator for the first term: Count(prefix_{k-1})
        # This is the total count of all k-grams starting with `current_prefix_tuple`.
        count_prefix_sum = self.prefix_total_counts[current_n_order].get(current_prefix_tuple, 0)

        # First term of Kneser-Ney formula: (max(0, Count(prefix_k-1, w_t) - delta)) / Count(prefix_k-1)
        first_term = 0.0
        if count_prefix_sum > 0:
            first_term = max(0, count_prefix_next_token - self.delta) / count_prefix_sum

        # Lambda term: (delta / Count(prefix_{k-1})) * N_1+(prefix_{k-1})
        # N_1+(prefix_{k-1}) is the number of unique words that follow prefix_{k-1} to form a k-gram.
        num_unique_next_words_for_prefix = self.N_1_plus_prefix[current_n_order].get(current_prefix_tuple, 0)

        lambda_term = 0.0
        if count_prefix_sum > 0:
            lambda_term = (self.delta / count_prefix_sum) * num_unique_next_words_for_prefix

        # Recursive call for lower-order model: P_kn(w_t | prefix_{k-2})
        # The prefix for the (current_n_order - 1) model is `current_prefix_tuple` without its first token.
        lower_order_prefix_tokens = list(current_prefix_tuple[1:])
        lower_order_prefix_str = ' '.join(lower_order_prefix_tokens)

        lower_order_prob = self._get_next_token_prob_recursive(lower_order_prefix_str, next_token, current_n_order - 1)

        return first_term + lambda_term * lower_order_prob

    def get_possible_next_tokens(self, prefix):
        """
        Returns a dictionary {token : it's probability} for all tokens with non-zero probabilities
        using Kneser-Ney smoothing. Iterates through the entire vocabulary to find probabilities.
        """
        probs = {}
        # With smoothing, almost all words in the vocabulary will have a non-zero probability.
        for token in self.vocab:
            prob = self.get_next_token_prob(prefix, token)
            if prob > 0:
                probs[token] = prob

        # Ensure UNK is handled explicitly if it's considered part of prediction space
        # (e.g. if the original text contained explicit UNK tokens)
        if UNK in self.vocab and UNK not in probs:
            prob_unk = self.get_next_token_prob(prefix, UNK)
            if prob_unk > 0: probs[UNK] = prob_unk

        return Counter(probs)

    def get_next_token_prob(self, prefix, next_token):
        """
        Public interface to calculate P_kn(next_token | prefix) using Kneser-Ney smoothing.
        This method initiates the recursive probability calculation.
        """
        return self._get_next_token_prob_recursive(prefix, next_token, self.n)


In [ ]:
dummy_lm = KneserNeyLanguageModel(dummy_lines, n=1)
sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab])

1.0

In [ ]:
dummy_lm.get_next_token_prob('a', "video")

0.0001580762405890272

In [ ]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [ ]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f, vocab: %i" % (n, ppx, len(lm.vocab)))

N = 1, Perplexity = 2885.69413, vocab: 54177
N = 2, Perplexity = 873.21065, vocab: 54177
N = 3, Perplexity = 246906203.92950, vocab: 54177


In [ ]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=0.75)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f, vocab: %i" % (n, ppx, len(lm.vocab)))

N = 1, Perplexity = 2885.69413, vocab: 54177
N = 2, Perplexity = 832.73936, vocab: 54177
N = 3, Perplexity = 231976900.22803, vocab: 54177


In [ ]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=0.5)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f, vocab: %i" % (n, ppx, len(lm.vocab)))

N = 1, Perplexity = 2885.69413, vocab: 54177
N = 2, Perplexity = 847.34819, vocab: 54177
N = 3, Perplexity = 259386919.80454, vocab: 54177


In [ ]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=0.25)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f, vocab: %i" % (n, ppx, len(lm.vocab)))

N = 1, Perplexity = 2885.69413, vocab: 54177
N = 2, Perplexity = 897.85584, vocab: 54177
N = 3, Perplexity = 330815048.26691, vocab: 54177
